# 🍔 Food Delivery Time Prediction using Linear Regression from Scratch
**Data Science Capstone / Portfolio Project**

This notebook presents an end-to-end Data Science project implementing **Ordinary Least Squares (OLS) Linear Regression from scratch** using pure NumPy, applied to predicting **Food Delivery Times** (inspired by platforms like UberEats, DoorDash, and Zomato).

### Key Highlights:
1. **Mathematical Formulation**: OLS cost function $J(\theta)$, Batch Gradient Descent updates, and closed-form Normal Equation.
2. **Custom Implementation**: Custom `LinearRegressionScratch`, `StandardScalerScratch`, and evaluation metrics.
3. **Comprehensive Comparison**: Direct parameter-by-parameter and metric-by-metric comparison against `scikit-learn.linear_model.LinearRegression`.
4. **Visual Diagnostics**: Convergence curves, residual analysis, homoscedasticity checks, and coefficient impact.

In [ ]:
# Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Scikit-learn baseline
from sklearn.linear_model import LinearRegression as SklearnLinearRegression

# Project custom modules
from linear_regression_scratch import (
    LinearRegressionScratch, 
    StandardScalerScratch, 
    train_test_split_scratch,
    mean_squared_error,
    root_mean_squared_error,
    mean_absolute_error,
    r2_score,
    adjusted_r2_score
)
from data_loader import load_and_preprocess_data

# Visual aesthetics
sns.set_theme(style="whitegrid", palette="deep")
%matplotlib inline

## 1. Load and Explore the Food Delivery Dataset
Let's load the dataset and inspect summary statistics.

In [ ]:
X_train, X_test, y_train, y_test, feature_names, scaler, raw_df = load_and_preprocess_data()
print(f"Total samples: {len(raw_df)}")
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")
raw_df.head()

In [ ]:
# Summary statistics
raw_df.describe()

## 2. Train Models
We train:
1. **Custom Scratch Model (Batch Gradient Descent)**
2. **Custom Scratch Model (Normal Equation)**
3. **Scikit-Learn `LinearRegression` (Benchmark)**

In [ ]:
# 1. Scratch Gradient Descent
scratch_gd = LinearRegressionScratch(learning_rate=0.05, n_iterations=3000, method='gradient_descent', tol=1e-8)
scratch_gd.fit(X_train, y_train)
print(f"Scratch GD converged in {scratch_gd.n_iter_} iterations ({scratch_gd.training_time_sec_*1000:.2f} ms)")

# 2. Scratch Normal Equation
scratch_norm = LinearRegressionScratch(method='normal_equation')
scratch_norm.fit(X_train, y_train)
print(f"Scratch Normal Equation solved in {scratch_norm.training_time_sec_*1000:.2f} ms")

# 3. Scikit-Learn LinearRegression
sk_lr = SklearnLinearRegression()
t0 = time.perf_counter()
sk_lr.fit(X_train, y_train)
sk_time = time.perf_counter() - t0
print(f"Scikit-Learn OLS fitted in {sk_time*1000:.2f} ms")

## 3. Parameter-by-Parameter Comparison
Let's compare the exact weights $w$ and bias $b$ learned by each model.

In [ ]:
params_df = pd.read_csv('model_parameters_comparison.csv')
params_df

## 4. Performance Metrics Comparison
Let's evaluate $R^2$, Adjusted $R^2$, RMSE, and MAE across models on the test set.

In [ ]:
metrics_df = pd.read_csv('model_metrics_comparison.csv')
metrics_df

## 5. Visual Diagnostics
Let's display the diagnostic plots generated by the pipeline.

In [ ]:
from IPython.display import Image, display
display(Image('plots/cost_convergence.png'))
display(Image('plots/actual_vs_predicted.png'))
display(Image('plots/residuals_distribution.png'))
display(Image('plots/coefficients_comparison.png'))
display(Image('plots/learning_rate_comparison.png'))

## 6. Live Interactive Prediction Demo
Predict delivery time for a new real-world customer order.

In [ ]:
order = {
    'distance_km': 7.2,
    'prep_time_min': 25.0,
    'traffic_level': 3,
    'weather_condition': 1,
    'delivery_partner_rating': 4.7,
    'delivery_partner_age': 29,
    'num_items': 4
}
order_arr = np.array([[order[f] for f in feature_names]])
order_scaled = scaler.transform(order_arr)

time_scratch = scratch_gd.predict(order_scaled)[0]
time_sklearn = sk_lr.predict(order_scaled)[0]

print(f"Predicted Delivery Time (Scratch Model): {time_scratch:.2f} mins")
print(f"Predicted Delivery Time (Scikit-Learn):  {time_sklearn:.2f} mins")
print(f"Absolute Discrepancy: {abs(time_scratch - time_sklearn):.6f} mins")